# VisualQA India — train and run the complete hybrid system

Enable a GPU and Internet, then attach the **RDD 2022** Kaggle dataset. This notebook trains YOLO, evaluates it, loads BLIP-2, and starts the fused Gradio application. Training can take hours.

In [ ]:
import os, subprocess, sys

GITHUB_REPO = "https://github.com/JeyanthRavi/visualqa-india.git"
PROJECT_DIR = "/kaggle/working/visualqa-india"
if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-training.txt"], check=True)
print("Project ready:", os.getcwd())

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle settings."
print(torch.cuda.get_device_name(0))
print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

In [ ]:
# Set False when /kaggle/working/visualqa_runs/best.pt already exists.
RUN_TRAINING = True
if RUN_TRAINING:
    subprocess.run([
        sys.executable, "train_rdd2022_yolo.py",
        "--dataset-root", "/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT",
        "--epochs", "60",
        "--imgsz", "768",
    ], check=True)

In [ ]:
# The training subprocess has exited, so its GPU memory is released.
os.environ["VISUALQA_YOLO_MODEL"] = "/kaggle/working/visualqa_runs/best.pt"
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
import app
detector = app.get_detector()
analyzer = app.get_analyzer()
assert detector is not None, "Trained YOLO best.pt was not found."
print("YOLO ready:", detector.model_path)
print("BLIP-2 ready on:", analyzer.device)
app.demo.queue(default_concurrency_limit=1).launch(share=True)